Step 1: Define File Path and Load Data
This step defines the file path for the 2018 property assessment dataset and checks if the file exists. If the file is found, it loads the dataset, removes any leading/trailing whitespace from column names, and handles missing values for consistency.

In [1]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

In [2]:
# Define the file path
file_path = r'C:\Users\sul19\Desktop\701 Project\Property Assessment Datasets\Property Assessment 2018 V1.xlsx'

# Check if the file exists before attempting to load it
if os.path.exists(file_path):
    print("File found! Proceeding to load the data.")
    
    # Attempt to load the dataset using 'openpyxl' engine
    df = pd.read_excel(file_path, engine='openpyxl', na_values=['', ' '])

    # Strip any leading/trailing whitespace from the column names (just in case)
    df.columns = df.columns.str.strip()
    
else:
    print(f"File not found at {file_path}. Please check the file path.")

File found! Proceeding to load the data.


Step 2: Standardize Values in OVERALL_COND Column
This step replaces shorthand values in the OVERALL_COND column with descriptive labels. It also removes non-breaking spaces and invisible characters, handling blank values by assigning "none." This ensures data consistency for further analysis.

In [3]:
# Replace specific values in the 'OVERALL_COND' column before grouping the data
df['OVERALL_COND'] = df['OVERALL_COND'].replace({
    'A': 'A - Average',
    'G': 'G - Good',
    'E': 'E - Excellent',
    'F': 'F - Fair',
    'P': 'P - Poor'
})

# Replace non-breaking spaces and other invisible characters in the overall condition column
df['OVERALL_COND'] = df['OVERALL_COND'].astype(str).str.replace('\u00A0', '').str.strip()

# Replace any blank or missing values with 'none'
df['OVERALL_COND'] = df['OVERALL_COND'].replace(r'^\s*$', 'none', regex=True)

# Replace specific values in OVERALL_COND
df['OVERALL_COND'] = df['OVERALL_COND'].replace({
    'AVG - Default - Average': 'A - Average',
    'EX - Excellent': 'E - Excellent'
}, regex=False)

Step 3: Group Data by ZIP_CODE and Summarize OVERALL_COND Counts
In this step, the data is grouped by ZIP code to count occurrences of each condition, creating a summary that helps us understand the distribution of housing conditions across different areas.

In [4]:
# Group data by 'ZIP_CODE' and count the occurrences of each condition in the column
overall_cond_summary = df.groupby('ZIP_CODE')['OVERALL_COND'].value_counts().unstack().fillna(0)

# Display the result of the analysis
print("Housing condition summary by ZIP code:")
#print(condition_summary)
print(overall_cond_summary)

Housing condition summary by ZIP code:
OVERALL_COND  A - Average  E - Excellent  F - Fair  G - Good  P - Poor  \
ZIP_CODE                                                                 
2090.0                0.0            0.0       0.0       0.0       0.0   
2108.0               39.0           93.0       3.0     123.0       0.0   
2109.0               20.0            1.0       1.0       2.0       0.0   
2110.0                0.0            0.0       0.0       0.0       0.0   
2111.0               12.0            0.0       8.0       2.0       0.0   
2112.0                0.0            0.0       0.0       0.0       0.0   
2113.0               87.0            5.0      11.0      27.0       1.0   
2114.0               89.0           29.0       8.0     127.0       2.0   
2115.0               84.0           11.0       2.0      77.0       0.0   
2116.0              167.0          107.0      16.0     196.0       5.0   
2118.0              162.0           59.0      18.0     190.0       5.0   

Step 4: Map Condition Labels to Numeric Values for Analysis
To enable quantitative analysis, this step maps condition labels to numeric scores. We then calculate the mean condition score for each ZIP code, providing insights into average housing conditions by area.

Step 5: Convert Mean Condition Scores to Descriptive Labels
This step maps numeric mean scores back to descriptive labels to make results more interpretable. Each ZIP code receives a condition label that represents the general state of housing conditions.

In [5]:
# Function to assign numerical values to conditions 
def condition_to_numeric(cond):
    mapping = {
        'E - Excellent': 5,
        'VG - Very Good': 4,
        'G - Good': 3.5,
        'A - Average': 3,
        'F - Fair': 2,
        'P - Poor': 1.5,
        'VP - Very Poor': 1,
        'US - Unsound': 0,
        
        
        'none': np.nan  # Treat 'none' as NaN for numerical purposes
    }
    return mapping.get(cond, np.nan)

# Apply the mapping to calculate average conditions
df['OVERALL_COND_NUM'] = df['OVERALL_COND'].apply(condition_to_numeric)

# Group by ZIP_CODE and calculate the mean, count, and standard deviation
condition_analysis = df.groupby('ZIP_CODE').agg(
    overall_cond_mean=('OVERALL_COND_NUM', 'mean'),
).reset_index()

# Display the summary
print("Housing condition analysis by ZIP code:")
print(condition_analysis)

def mean_to_condition(mean_score):
    if mean_score >= 4.75:
        return 'Excellent'
    elif mean_score >= 4:
        return 'Very Good'
    elif mean_score >= 3.5:
        return 'Good'
    elif mean_score >= 3:
        return 'Average'
    elif mean_score >= 2:
        return 'Fair'
    elif mean_score >= 1:
        return 'Poor'
    elif mean_score >= 0.5:
        return 'Very Poor'
    
    else:
        return 'Unsound'

# Apply the function to map the means back to descriptive condition labels
condition_analysis['overall_cond_label'] = condition_analysis['overall_cond_mean'].apply(mean_to_condition)

# Print the result for each ZIP code
print("Overall Condition Analysis by ZIP Code:")
print(condition_analysis[['ZIP_CODE', 'overall_cond_mean', 'overall_cond_label']])

Housing condition analysis by ZIP code:
    ZIP_CODE  overall_cond_mean
0     2090.0                NaN
1     2108.0           3.947674
2     2109.0           3.083333
3     2110.0                NaN
4     2111.0           2.681818
5     2112.0                NaN
6     2113.0           3.083969
7     2114.0           3.433333
8     2115.0           3.336207
9     2116.0           3.587576
10    2118.0           3.432028
11    2119.0           3.078191
12    2120.0           3.107091
13    2121.0           3.059748
14    2122.0           3.044270
15    2124.0           3.065054
16    2125.0           3.063515
17    2126.0           3.031579
18    2127.0           3.097242
19    2128.0           3.057763
20    2129.0           3.185513
21    2130.0           3.140364
22    2131.0           3.050413
23    2132.0           3.075152
24    2133.0                NaN
25    2134.0           3.029107
26    2135.0           3.039575
27    2136.0           3.038485
28    2137.0                NaN


Step 6: Standardize YR_BUILT and YR_REMODEL Columns
This step ensures YR_BUILT and YR_REMODEL columns are numeric and replaces any years beyond 2024 with NaN, avoiding future-dated entries. We then calculate mean construction and remodel years by ZIP code for further analysis.

Step 7: Classify Buildings by Age
This step classifies buildings based on their construction or remodel year as 'Old,' 'Average,' or 'New.' This provides additional insights into the age distribution within each ZIP code.

In [6]:
# Ensure YR_BUILT and YR_REMODEL are numeric and replace years > 2024 with NaN
df['YR_BUILT'] = pd.to_numeric(df['YR_BUILT'], errors='coerce')
df['YR_REMODEL'] = pd.to_numeric(df['YR_REMODEL'], errors='coerce')
df['YR_BUILT'] = df['YR_BUILT'].apply(lambda x: x if x <= 2024 else np.nan)
df['YR_REMODEL'] = df['YR_REMODEL'].apply(lambda x: x if x <= 2024 else np.nan)

# Group by ZIP_CODE and calculate the mean for YR_BUILT and YR_REMODEL
condition_analysis = df.groupby('ZIP_CODE').agg(
    yr_built_mean=('YR_BUILT', 'mean'),
    yr_remodel_mean=('YR_REMODEL', 'mean')
).reset_index()

# Define thresholds for building classification
def classify_building(yr_built, yr_remodel, old_threshold=1970, new_threshold=2000):
    """
    Classify building as 'Old', 'Average', or 'New' based on YR_REMODEL or YR_BUILT.
    If YR_REMODEL exists, use it; otherwise, use YR_BUILT.
    """
    if not pd.isna(yr_remodel):
        year = yr_remodel  # Use YR_REMODEL if available
    else:
        year = yr_built  # Otherwise, use YR_BUILT
    
    if pd.isna(year):
        return 'Unknown'
    elif year <= old_threshold:
        return 'Old'
    elif year >= new_threshold:
        return 'New'
    else:
        return 'Average'
    
# Apply classification based on YR_BUILT and YR_REMODEL
condition_analysis['building_classification'] = condition_analysis.apply(
    lambda row: classify_building(row['yr_built_mean'], row['yr_remodel_mean']), axis=1)

# Print the classification results based on the YR_BUILT and YR_REMODEL means
print("Building Classification Based on YR_BUILT and YR_REMODEL:")
print(condition_analysis[['ZIP_CODE', 'yr_built_mean', 'yr_remodel_mean', 'building_classification']])

Building Classification Based on YR_BUILT and YR_REMODEL:
    ZIP_CODE  yr_built_mean  yr_remodel_mean building_classification
0     2090.0    1988.000000      1988.000000                 Average
1     2108.0    1546.598653      1910.212682                     Old
2     2109.0    1752.328797      1598.831212                     Old
3     2110.0    1618.928040      1095.675113                     Old
4     2111.0    1711.341707      1847.866283                     Old
5     2112.0       0.000000              NaN                     Old
6     2113.0    1813.413824      1903.450026                     Old
7     2114.0    1659.192899      1837.379484                     Old
8     2115.0    1661.702275      1874.662989                     Old
9     2116.0    1818.259732      1860.805870                     Old
10    2118.0    1794.060062      1898.906652                     Old
11    2119.0    1458.149256      1265.689907                     Old
12    2120.0    1591.524164      1236.208170 

Step 8: Include Address Details and Finalize Output
This final step incorporates street address details, merges overall condition labels, assigns a constant YEAR value, and saves the output as an Excel file for further use.

In [7]:
# Assuming 'ST_NUM' and 'ST_NAME' are part of the original dataset
# Extract those columns from the original dataset
st_num_name = df[['ZIP_CODE', 'ST_NUM', 'ST_NAME']].drop_duplicates()

# Merge 'st_num_name' with 'condition_analysis' to include 'ST_NUM' and 'ST_NAME' with the results
condition_analysis = pd.merge(condition_analysis, st_num_name, on='ZIP_CODE', how='left')

# Add overall condition label based on previous analysis
# Ensure that the column 'overall_cond_label' from earlier condition analysis is merged correctly
overall_cond_summary = df.groupby('ZIP_CODE').agg(
    overall_cond_mean=('OVERALL_COND_NUM', 'mean'),
).reset_index()

# Reapply the condition label mapping function to map numeric values to condition labels
def mean_to_condition(mean_score):
    if mean_score >= 4.75:
        return 'Excellent'
    elif mean_score >= 4:
        return 'Very Good'
    elif mean_score >= 3.5:
        return 'Good'
    elif mean_score >= 3:
        return 'Average'
    elif mean_score >= 2:
        return 'Fair'
    elif mean_score >= 1:
        return 'Poor'
    elif mean_score >= 0.5:
        return 'Very Poor'
    else:
        return 'Unsound'

overall_cond_summary['overall_cond_label'] = overall_cond_summary['overall_cond_mean'].apply(mean_to_condition)

# Merge overall condition labels with the condition_analysis DataFrame
condition_analysis = pd.merge(condition_analysis, overall_cond_summary[['ZIP_CODE', 'overall_cond_label']], on='ZIP_CODE', how='left')

# Assign a constant value for 'YEAR'
condition_analysis['YEAR'] = 2018

# Rearranging the columns as requested
final_output = condition_analysis[['YEAR', 'ST_NUM', 'ST_NAME', 'ZIP_CODE', 'overall_cond_label', 'building_classification']]

# Save the final result to a new Excel file
output_file_path = 'Property_Assessment_2018_Output.xlsx'
final_output.to_excel(output_file_path, index=False)

print(f"File saved successfully to {output_file_path}")

File saved successfully to Property_Assessment_2018_Output.xlsx
